In [24]:
import pandas as pd
import shutil

In [ ]:
# Get the Fine-tuned BERT Clssifier here

from huggingface_hub import HfApi

api = HfApi(token=os.getenv("HF_TOKEN"))
api.upload_folder(
    folder_path="/path/to/local/model",
    repo_id="Ruslan1995/bert_classifier_closed_llm_vs_humans",
    repo_type="model",
)

In [ ]:
!pip install scikit-learn

In [ ]:
!pip install datasets

In [ ]:
!pip install transformers
!pip install torch

In [2]:
!pip install numpy==1.26.4

In [3]:
import numpy as np

In [4]:
print(np.__version__)

1.26.4


In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import transformers
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from datasets import Dataset, DatasetDict

2025-05-28 08:23:05.794600: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748420585.817884     109 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748420585.824874     109 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [27]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API")
secret_value_1 = user_secrets.get_secret("HF_API_WRITE")

In [7]:
import os
os.environ["WANDB_API_KEY"] = secret_value_0

In [8]:
# For local dataset
df = pd.read_json('/kaggle/input/ai-dataset/data_correct.json')

In [9]:
# df = pd.read_json('drive/MyDrive/ai_project/data_correct.json')
df

,text,label,domain
0,- Strengths:\n* Outperforms ALIGN in supervise...,human_text,human
1,the mention or also includes other related ent...,machine_text,chatgpt
2,This paper addresses the problem of disambigua...,human_text,human
3,For entities: their profiles consist of neighb...,machine_text,chatgpt
4,"- Strengths:\nGood ideas, simple neural learni...",human_text,human
...,...,...,...
61792,Bobby Douglas also suggested an American woman...,human_text,human
61793,"Ursula von der Leyen, a close ally of Chancell...",human_text,human
61794,The hosts' display was uncertain and their opp...,machine_text,Qwen
61795,Birkhoff-Smale theorem says that transverse ho...,machine_text,Llama3


In [10]:
df['domain'].value_counts()

domain
human      34713
Llama2     17214
chatgpt     4154
Llama3      1966
GPT3.5      1882
Qwen        1868
Name: count, dtype: int64

Task 1 - closed models VS humans + open models
Task 2 - closed models VS humans only

In [11]:
label_mapping_task_1 = {
    'human': 0,
    'chatgpt': 1,
    'GPT3.5': 1,
    'Llama2': 0,
    'Llama3': 0,
    'Qwen': 0
}

label_mapping_task_2 = {
    'human': 0,
    'chatgpt': 1,
    'GPT3.5': 1,
    'Llama2': -1,
    'Llama3': -1,
    'Qwen': -1
}

In [12]:
df_a = df.copy()
df_a['target'] = df_a['domain'].map(label_mapping_task_1)
df_a

,text,label,domain,target
0,- Strengths:\n* Outperforms ALIGN in supervise...,human_text,human,0
1,the mention or also includes other related ent...,machine_text,chatgpt,1
2,This paper addresses the problem of disambigua...,human_text,human,0
3,For entities: their profiles consist of neighb...,machine_text,chatgpt,1
4,"- Strengths:\nGood ideas, simple neural learni...",human_text,human,0
...,...,...,...,...
61792,Bobby Douglas also suggested an American woman...,human_text,human,0
61793,"Ursula von der Leyen, a close ally of Chancell...",human_text,human,0
61794,The hosts' display was uncertain and their opp...,machine_text,Qwen,0
61795,Birkhoff-Smale theorem says that transverse ho...,machine_text,Llama3,0


In [13]:
df_b = df.copy()[df['domain'].isin(['human', 'chatgpt', 'GPT3.5'])]
df_b['target'] = df_b['domain'].map(label_mapping_task_2)
df_b

,text,label,domain,target
0,- Strengths:\n* Outperforms ALIGN in supervise...,human_text,human,0
1,the mention or also includes other related ent...,machine_text,chatgpt,1
2,This paper addresses the problem of disambigua...,human_text,human,0
3,For entities: their profiles consist of neighb...,machine_text,chatgpt,1
4,"- Strengths:\nGood ideas, simple neural learni...",human_text,human,0
...,...,...,...,...
61790,"The classical theorem of Moser, proven in , es...",human_text,human,0
61791,New figures from Police Scotland show the numb...,human_text,human,0
61792,Bobby Douglas also suggested an American woman...,human_text,human,0
61793,"Ursula von der Leyen, a close ally of Chancell...",human_text,human,0


In [ ]:
# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# def tokenize(sample):
#   return tokenizer(sample['text'], truncation=True, padding='max_length', max_length=512)

In [ ]:
# df_a = df_a.rename(columns={"target": "labels"})

# train_df_a, test_df_a = train_test_split(df_a, test_size=0.15, stratify=df_a['labels'], random_state=42)

# train_ds_a = Dataset.from_pandas(train_df_a)
# test_ds_a = Dataset.from_pandas(test_df_a)

# dataset = DatasetDict({
#     'train': train_ds_a,
#     'test': test_ds_a
# })

# dataset = dataset.map(tokenize, batched=True)
# dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

In [15]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    eval_strategy="epoch",
    logging_dir="/kaggle/working/logs",
    logging_steps=10,
    logging_strategy="steps",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="wandb",
    run_name="bert-closed-vs-human",
    metric_for_best_model="f1",
    disable_tqdm=False
)

In [16]:
def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    labels = pred.label_ids
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

In [ ]:
# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=dataset["train"],
#     eval_dataset=dataset["test"],
#     compute_metrics=compute_metrics,
# )
# trainer.train()

In [ ]:
# trainer.save_model("bert_classifier_a")
# tokenizer.save_pretrained("bert_classifier_a")

In [17]:
tokenizer_b = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_b(sample):
  return tokenizer_b(sample['text'], truncation=True, padding='max_length', max_length=512)

In [18]:
df_b = df_b.rename(columns={"target": "labels"})

train_df_b, test_df_b = train_test_split(df_b, test_size=0.15, stratify=df_b['labels'], random_state=42)

train_ds_b = Dataset.from_pandas(train_df_b)
test_ds_b = Dataset.from_pandas(test_df_b)

dataset_b = DatasetDict({
    'train': train_ds_b,
    'test': test_ds_b
})

dataset_b = dataset_b.map(tokenize_b, batched=True)
dataset_b.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/34636 [00:00<?, ? examples/s]

Map:   0%|          | 0/6113 [00:00<?, ? examples/s]

In [19]:
model_b = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
trainer_b = Trainer(
    model=model_b,
    args=training_args,
    train_dataset=dataset_b["train"],
    eval_dataset=dataset_b["test"],
    compute_metrics=compute_metrics,
)
trainer_b.train()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: ruslan-iskhak94 (ruslan-iskhak94-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.000000,0.026494,0.995420,0.996602,0.972376,0.984340
2,0.000000,0.015227,0.997219,0.992239,0.988950,0.990592
3,0.000000,0.005979,0.999182,1.000000,0.994475,0.997230


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


TrainOutput(global_step=6495, training_loss=0.017677275073824158, metrics={'train_runtime': 6712.7723, 'train_samples_per_second': 15.479, 'train_steps_per_second': 0.968, 'total_flos': 2.733934354034688e+16, 'train_loss': 0.017677275073824158, 'epoch': 3.0})

In [22]:
trainer_b.evaluate(dataset_b["test"])

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.005979499779641628,
 'eval_accuracy': 0.9991820709962376,
 'eval_precision': 1.0,
 'eval_recall': 0.994475138121547,
 'eval_f1': 0.9972299168975068,
 'eval_runtime': 119.3402,
 'eval_samples_per_second': 51.223,
 'eval_steps_per_second': 3.209,
 'epoch': 3.0}

In [23]:
trainer_b.save_model("/kaggle/working/bert_classifier_b")
tokenizer_b.save_pretrained("/kaggle/working/bert_classifier_b")

('/kaggle/working/bert_classifier_b/tokenizer_config.json',
 '/kaggle/working/bert_classifier_b/special_tokens_map.json',
 '/kaggle/working/bert_classifier_b/vocab.txt',
 '/kaggle/working/bert_classifier_b/added_tokens.json',
 '/kaggle/working/bert_classifier_b/tokenizer.json')

In [25]:
shutil.make_archive("bert_classifier_b", "zip", "/kaggle/working/bert_classifier_b")

'/kaggle/working/bert_classifier_b.zip'

In [26]:
!pip install huggingface_hub

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [30]:
from huggingface_hub import login, create_repo, upload_folder
login(secret_value_1)

In [31]:
hf_username = "Ruslan1995"  # ← твой никнейм на HF
repo_name = "bert_classifier_closed_llm_vs_humans"  # любое уникальное название
model_dir = "/kaggle/working/bert_classifier_b"  # путь к папке модели

# Создаем репозиторий (если уже есть — не страшно)
create_repo(f"{hf_username}/{repo_name}", exist_ok=True)

# Загружаем модель
upload_folder(
    folder_path=model_dir,
    repo_id=f"{hf_username}/{repo_name}",
    commit_message="Upload trained BERT classifier",
    repo_type="model"
)

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.30k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Ruslan1995/bert_classifier_closed_llm_vs_humans/commit/93cc7a832775fd5ebbca72595edf1d51020966d8', commit_message='Upload trained BERT classifier', commit_description='', oid='93cc7a832775fd5ebbca72595edf1d51020966d8', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Ruslan1995/bert_classifier_closed_llm_vs_humans', endpoint='https://huggingface.co', repo_type='model', repo_id='Ruslan1995/bert_classifier_closed_llm_vs_humans'), pr_revision=None, pr_num=None)